# Substrate Telemetry Helpers

> Shared GPU/CPU attribution helpers used by both `JobQueue._sample_resource_snapshot` (CR-6 Stage 3) and `CapabilityManager._record_sample_safe` (CR-7).

The substrate's worker subprocess may spawn further subprocesses (e.g. Voxtral-vLLM launches an OpenAI-compatible vLLM server as a *grandchild* of the substrate). Per-PID GPU attribution that matches only the worker's own PID misses those grandchildren, so the substrate underreports GPU usage for any subprocess-spawning plugin.

The fix is to attribute GPU memory across the **worker's process subtree**: the worker's `/stats` endpoint reports its subtree PIDs (it already walks `proc.children(recursive=True)` for the RAM sum, so the PID list is essentially free), and the substrate intersects that set with the system-monitor plugin's `list_processes()` enumeration of GPU-holding PIDs.

In [ ]:
#| default_exp core._telemetry

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Dict, Iterable, Optional

## `attribute_gpu_to_worker_subtree`

Sums GPU memory across the worker's subtree by intersecting the worker-reported subtree PID set with the system-monitor's GPU-process enumeration. Returns `None` when no system-monitor is available; returns `(0.0, None)` when sysmon is available but no subtree PID holds GPU memory. The `gpu_index` returned is taken from the highest-VRAM PID in the subtree (a stable tiebreak; multi-GPU subtree-spanning workloads are exceptional and can refine later).

In [ ]:
#| export
def _proc_field(proc: Any, key: str, default: Any = None) -> Any:
    """Read a field from a sysmon process record, accepting dict or dataclass.

    `MonitorPlugin.list_processes()` returns `ProcessStats` dataclasses (CR-3),
    but proxy round-trips frequently coerce to dicts. Accept both so the
    helper works against either form without the caller pre-normalizing.
    """
    if isinstance(proc, dict):
        return proc.get(key, default)
    return getattr(proc, key, default)


def _worker_subtree_pids(stats: Dict[str, Any]) -> set:
    """Build the worker subtree PID set from a `/stats` dict.

    Falls back to a single-pid set when `subtree_pids` is absent (pre-fix
    workers, mock test fixtures). The worker pid itself is always included.
    """
    tree: set = set()
    worker_pid = stats.get('pid')
    if worker_pid is not None:
        tree.add(worker_pid)
    for sub_pid in (stats.get('subtree_pids') or ()):
        if sub_pid is not None:
            tree.add(sub_pid)
    return tree


def attribute_gpu_to_worker_subtree(
    stats: Dict[str, Any],  # Worker `/stats` payload (must include 'pid'; uses 'subtree_pids' if present)
    sysmon: Any,            # The configured MonitorPlugin (or None)
) -> Optional[Dict[str, Any]]:
    """Attribute GPU memory across the worker's process subtree.

    Returns `{'gpu_memory_mb': float, 'gpu_index': Optional[int]}` when sysmon
    is reachable, or `None` when sysmon isn't configured / doesn't expose
    `list_processes()` / errors out. Callers treat `None` as "sysmon
    unavailable" and leave GPU snapshot fields as their defaults; a 0.0 sum
    means sysmon worked but no subtree PID holds GPU memory (CPU-only plugin
    on a GPU box).
    """
    if sysmon is None or not hasattr(sysmon, 'list_processes'):
        return None
    try:
        procs = sysmon.list_processes() or ()
    except Exception:
        return None

    tree = _worker_subtree_pids(stats)
    if not tree:
        return {'gpu_memory_mb': 0.0, 'gpu_index': None}

    total_mb: float = 0.0
    best_pid_mb: float = -1.0
    best_gpu_index: Optional[int] = None
    for proc in procs:
        pid = _proc_field(proc, 'pid')
        if pid is None or pid not in tree:
            continue
        mem = _proc_field(proc, 'gpu_memory_mb', 0.0) or 0.0
        try:
            mem = float(mem)
        except (TypeError, ValueError):
            mem = 0.0
        total_mb += mem
        if mem > best_pid_mb:
            best_pid_mb = mem
            best_gpu_index = _proc_field(proc, 'gpu_index')

    return {'gpu_memory_mb': total_mb, 'gpu_index': best_gpu_index}

## Tests

In [ ]:
# Sysmon unavailable → None (substrate leaves GPU snapshot fields at defaults).
stats = {'pid': 1111, 'subtree_pids': [2222]}
assert attribute_gpu_to_worker_subtree(stats, None) is None

class _NoListProcesses:
    def get_system_status(self): return {}

assert attribute_gpu_to_worker_subtree(stats, _NoListProcesses()) is None

In [ ]:
# Worker pid 1111 spawned vLLM grandchild 9999; sysmon enumerates the grandchild.
# The pre-fix substrate matched only the worker pid and reported gpu_memory_mb=0.
class _Sysmon:
    def list_processes(self):
        return [
            {'pid': 9999, 'gpu_index': 0, 'gpu_memory_mb': 4096.0, 'command': 'vllm server'},
            {'pid': 7777, 'gpu_index': 0, 'gpu_memory_mb': 512.0, 'command': 'other process'},
        ]

stats = {'pid': 1111, 'subtree_pids': [1111, 9999]}
result = attribute_gpu_to_worker_subtree(stats, _Sysmon())
assert result == {'gpu_memory_mb': 4096.0, 'gpu_index': 0}, result

In [ ]:
# Worker subtree spans multiple GPU-holding PIDs; total sums + best gpu_index is the highest-VRAM PID's.
class _Sysmon2:
    def list_processes(self):
        return [
            {'pid': 100, 'gpu_index': 0, 'gpu_memory_mb': 256.0},
            {'pid': 200, 'gpu_index': 1, 'gpu_memory_mb': 1024.0},
            {'pid': 300, 'gpu_index': 1, 'gpu_memory_mb': 512.0},  # in tree but smaller
        ]

stats = {'pid': 100, 'subtree_pids': [100, 200, 300]}
result = attribute_gpu_to_worker_subtree(stats, _Sysmon2())
assert result == {'gpu_memory_mb': 256.0 + 1024.0 + 512.0, 'gpu_index': 1}, result

In [ ]:
# CPU-only plugin on a GPU box: sysmon works but the worker's subtree holds no GPU memory.
# Returns 0.0 (not None) so empirical store records an honest "no GPU usage" sample.
class _Sysmon3:
    def list_processes(self):
        return [{'pid': 9999, 'gpu_index': 0, 'gpu_memory_mb': 4096.0}]

stats = {'pid': 1111, 'subtree_pids': [1111, 2222]}
result = attribute_gpu_to_worker_subtree(stats, _Sysmon3())
assert result == {'gpu_memory_mb': 0.0, 'gpu_index': None}, result

In [ ]:
# Accept dataclass-shaped ProcessStats (CR-3 worker-direct calls) in addition to dicts (proxy round-trip).
from dataclasses import dataclass

@dataclass
class _PS:
    pid: int
    gpu_index: int
    gpu_memory_mb: float
    command: str = ''

class _SysmonDc:
    def list_processes(self):
        return [_PS(pid=9999, gpu_index=0, gpu_memory_mb=4096.0)]

stats = {'pid': 1111, 'subtree_pids': [9999]}
result = attribute_gpu_to_worker_subtree(stats, _SysmonDc())
assert result == {'gpu_memory_mb': 4096.0, 'gpu_index': 0}, result

In [ ]:
# Backward compat: pre-fix worker /stats without `subtree_pids` falls back to {worker_pid} only.
# (Substrate publishing precedes the cascade; in-between, hosts may have an old worker but a new substrate.)
class _Sysmon4:
    def list_processes(self):
        return [
            {'pid': 1111, 'gpu_index': 0, 'gpu_memory_mb': 256.0},
            {'pid': 9999, 'gpu_index': 0, 'gpu_memory_mb': 4096.0},  # grandchild — invisible without subtree_pids
        ]

stats = {'pid': 1111}  # no subtree_pids
result = attribute_gpu_to_worker_subtree(stats, _Sysmon4())
assert result == {'gpu_memory_mb': 256.0, 'gpu_index': 0}, result  # worker-only, the pre-fix behavior

In [ ]:
# Sysmon error → None (failures shouldn't break snapshot/sample paths).
class _SysmonBroken:
    def list_processes(self):
        raise RuntimeError('sysmon explosion')

stats = {'pid': 1111, 'subtree_pids': [1111]}
assert attribute_gpu_to_worker_subtree(stats, _SysmonBroken()) is None

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()